Esta etapa silver toma la data de bronze y mejora la calidad de la información eliminando duplicados con un procesamiento incremental. 

Inferimos el esquema del json

In [0]:
ruta_origen = "iol_challenge.bronze.raw_ingestion"
ruta_destino = "iol_challenge.silver.deduped_transactions"

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

data_sample = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "transaction") \
    .withColumn("schema", schema_of_json(col("data"))) \
    .select("schema").first()[0]

In [0]:
data_sample

In [0]:
from pyspark.sql.functions import col, row_number, max as spark_max, length, md5, concat_ws, lit, explode, from_json
from pyspark.sql.window import Window
from delta.tables import DeltaTable



schema = """
STRUCT<fecha: STRING, 
tipoTran: STRING,
id_cliente: STRING, 
descripcion_titulo: STRING, 
moneda: STRING, 
simbolo_titulo: STRING, 
cantidad: BIGINT, 
precio: DOUBLE, 
id_transaccion: STRING, 
origen: STRING>
"""

df_origen_raw = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "transaction") \
    .withColumn("parsed_dict", from_json(col("data"), schema)) \
    .withColumn("sk_transaccion", md5(concat_ws(lit("||"), col("parsed_dict.id_transaccion")))) \
    .select(
        col("sk_transaccion"),
        col("parsed_dict.*"), 
        col("dia"),
        col("anio"),
        col("mes"),
        col("timestamp_ejecucion"),
        col("errores_calidad"),
        col("tiene_errores_calidad"),
    )

# Tomamos la última versión ingestada de la transacción preferentemente sin errores de calidad
ventana_dedup = Window.partitionBy("id_transaccion").orderBy(col("tiene_errores_calidad").asc(), col("timestamp_ejecucion").desc())

df_lote_deduplicado = df_origen_raw \
    .withColumn("row_num", row_number().over(ventana_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \


if spark.catalog.tableExists(ruta_destino):
    tabla_destino = DeltaTable.forName(spark, ruta_destino)
    
    max_timestamp = tabla_destino.toDF() \
        .select(spark_max("timestamp_ejecucion")) \
        .collect()[0][0]
    
    if max_timestamp is not None:
        df_lote_filtrado = df_lote_deduplicado.filter(col("timestamp_ejecucion") >= max_timestamp)
    else:
        df_lote_filtrado = df_lote_deduplicado

    tabla_destino.alias("target") \
        .merge(
            df_lote_filtrado.alias("source"),
            "target.id_transaccion = source.id_transaccion"
        ) \
        .whenMatchedUpdateAll(
            condition="source.timestamp_ejecucion > target.timestamp_ejecucion"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

else:
    df_lote_deduplicado.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ruta_destino)

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_transactions limit 10;


In [0]:
%sql
SELECT id_transaccion, COUNT(*) FROM iol_challenge.silver.deduped_transactions group by id_transaccion
    having count(*)>1;